# Faz 2 — Veri temizliği (data cleaning)

**Bu notebook'ta ne yapacağız?** Ham veriden (`data/`) analize hazır, tek bir temiz dosya üreteceğiz: `clean/countries_1950_2100.csv`.

**Neden?** Faz 1'de veride üç tuzak bulduk: toplam satırlar (çifte sayım), 2101'in boş olması ve `DoublingTime`'ın yapısal eksikliği. Bu tuzakları **bir kez, belgelenmiş şekilde** temizlersek sonraki bütün fazlar aynı temiz dosyadan başlar. Buna **tekrarlanabilirlik (reproducibility)** denir.

**Nasıl çalışacaksın?**
1. Hücreleri **yukarıdan aşağıya, tek tek** çalıştır (**Shift+Enter**).
2. Her kod hücresinin altında bir **kontrol (assert)** var. Hata vermezse kontrol geçti demektir.
3. 🟡 işaretli yerlerde **durup soruyu cevapla**. Cevaplarını sohbete yaz.

| Adım | İş |
|---|---|
| 1 | Veriyi oku |
| 2 | Sadece ülkeleri al (237) |
| 3 | 2101'i çıkar |
| 4 | Gereksiz sütunları ve `DoublingTime`'ı çıkar, bölge adlarını ekle |
| 5 | `donem` sütunu ekle (tahmin / projeksiyon) |
| 6 | Ek dosyayla birleştir (merge) |
| 7 | Dünya toplamıyla kontrol et |
| 8 | **K1 kararı:** ülke kapsamı |
| 9 | Kaydet |

## Adım 1 — Veriyi oku
- `m`: Ana dosya (Medium). Faz 1'deki ile aynı.
- `sup`: Ek dosya (Supplement). İçinde `LE60` (60 yaşında kalan yaşam beklentisi) ve cinsiyete göre doğum sayıları var.

In [1]:
import pandas as pd
import numpy as np

m = pd.read_csv("../data/WPP2024_Demographic_Indicators_Medium_csv.gz",
                encoding="utf-8-sig", low_memory=False)
sup = pd.read_csv("../data/WPP2024_Supplement_Births_and_LE60.csv",
                  encoding="utf-8-sig")

print("Medium :", m.shape)
print("Ek dosya:", sup.shape)

Medium : (84360, 67)
Ek dosya: (35636, 9)


**Satır satır:**
- `encoding="utf-8-sig"`: Dosyanın başındaki görünmez BOM işaretini atar. Böylece ilk sütunun adı bozuk okunmaz.
- `low_memory=False`: Pandas'ın dosyayı parça parça okurken sütun tipini yanlış tahmin etmesini önler.
- `.shape`: (satır, sütun) sayısını verir. Beklenen: Medium **(84360, 67)**, ek dosya **(35636, 9)**.

## Adım 2 — Sadece ülkeleri al

**Neden?** Dosyada ülkelerin yanında "World", "Western Asia", "NATO" gibi **toplam satırları** da var. Hepsini toplarsak aynı insanları defalarca sayarız. Faz 1'de bunu gördük: Tüm satırların toplamı 362 milyar çıktı, gerçek dünya nüfusu ise 8 milyar.

**Nasıl?** `LocTypeName` sütunu her satırın ne tür bir yer olduğunu söylüyor. Sadece `"Country/Area"` olanları tutuyoruz.

In [2]:
ulke = m[m["LocTypeName"] == "Country/Area"].copy()

print("Ülke sayısı (LocID):", ulke["LocID"].nunique())
print("Satır sayısı       :", len(ulke))

# Kontroller
assert ulke["LocID"].nunique() == 237
assert len(ulke) == 237 * 152              # her ülke 1950–2101 = 152 yıl
assert ulke["ISO3_code"].notna().all()     # her ülkenin ISO3 kodu var

Ülke sayısı (LocID): 237
Satır sayısı       : 36024


**Satır satır:**
- `m[m["LocTypeName"] == "Country/Area"]`: Koşulu sağlayan satırları seçer.
- `.copy()`: Seçilen satırları **bağımsız yeni bir tablo** olarak kopyalar. Kopyalamazsak, sonradan sütun eklerken pandas "SettingWithCopyWarning" uyarısı verir. Bunun sebebi, `ulke`'nin hâlâ `m`'ye bağlı bir görüntü (view) olabilmesidir.
- `assert koşul`: "Bu koşul doğru olmalı" demektir. Doğruysa sessizce geçer, yanlışsa **hata verip durur**. Temizlikte her adımı böyle kontrol ediyoruz.
  - Böylece bir şey yanlış giderse hata hemen görünür, analizin ortasında fark edilmez.

## Adım 3 — 2101 yılını çıkar

**Neden?** 2101 satırlarında sadece `TPopulation1Jan` dolu. Sebebi: 1 Ocak 2101, 2100 yılının bitişi demek. Diğer göstergeler boş. Analiz aralığımız **1950–2100** (151 yıl).

In [3]:
ulke = ulke[ulke["Time"] <= 2100]

print("Yıl aralığı:", ulke["Time"].min(), "–", ulke["Time"].max())
print("Satır sayısı:", len(ulke))

assert len(ulke) == 237 * 151

Yıl aralığı: 1950 – 2100
Satır sayısı: 35787


## Adım 4 — Sütunları düzenle ve bölge adlarını ekle

**Çıkaracaklarımız:**
- **`DoublingTime`:** Yapısal eksiklik içeriyor ve `PopGrowthRate`'in birebir dönüşümü. Yeni bilgi taşımıyor.
- **Yalnızca tanımlama ve sıralama için olan sütunlar:** `SortOrder` (UN'in listeleme sırası), `Notes` (dipnot numarası), `ISO2_code`, `SDMX_code` (başka kod sistemleri), `LocTypeID` ve `LocTypeName` (artık hepsi "ülke"), `VarID` ve `Variant` (hepsi "Medium").

**Ekleyeceklerimiz:** Her ülkenin **alt bölgesi** (ör. Türkiye → Western Asia) ve **bölgesi** (ör. → Asia).
- **Neden?** EDA'da ve kümelemede "ülkeler bölgelere göre mi ayrışıyor?" diye soracağız. Bu soruyu sorabilmek için bölge adları gerekli.

**Nasıl?** `ParentID`, ülkenin bağlı olduğu alt bölgenin numarası. O alt bölgenin de kendi `ParentID`'si var, o da bölgeyi gösteriyor. İki adımda yukarı çıkacağız: ülke → alt bölge → bölge.

In [4]:
# 1) Her yerin (LocID) adını ve üst yerini tutan küçük bir sözlük tablosu
yerler = m.drop_duplicates("LocID").set_index("LocID")[["Location", "ParentID"]]

# 2) Ülke → alt bölge → bölge
ulke["altbolge"] = ulke["ParentID"].map(yerler["Location"])
bolge_id         = ulke["ParentID"].map(yerler["ParentID"])
ulke["bolge"]    = bolge_id.map(yerler["Location"])

# 3) Bölgesi bulunamayan ülke var mı?
bulunamayan = ulke.loc[ulke["altbolge"].isna(), ["Location", "ParentID"]].drop_duplicates()
print("Bölgesi bulunamayan ülkeler:")
print(bulunamayan.to_string(index=False))
print()

# Bu 5 ülkenin ParentID'si 918, ama 918 numaralı yer dosyada yok (bkz. açıklama).
# UN'de bu ülkeler "Northern America" bölgesinde (LocID 905). Elle dolduruyoruz.
ulke.loc[ulke["ParentID"] == 918, ["altbolge", "bolge"]] = "Northern America"

# 4) Gereksiz sütunları çıkar
atilacak = ["DoublingTime", "SortOrder", "Notes", "ISO2_code", "SDMX_code",
            "LocTypeID", "LocTypeName", "VarID", "Variant"]
ulke = ulke.drop(columns=atilacak)

print(ulke.loc[ulke["Location"] == "Türkiye", ["Location", "altbolge", "bolge"]].head(1))
print()
print(ulke.drop_duplicates("LocID")["bolge"].value_counts())

assert ulke["altbolge"].notna().all() and ulke["bolge"].notna().all()

Bölgesi bulunamayan ülkeler:
                 Location  ParentID
                  Bermuda     918.0
                   Canada     918.0
                Greenland     918.0
Saint Pierre and Miquelon     918.0
 United States of America     918.0

      Location      altbolge bolge
62016  Türkiye  Western Asia  Asia

bolge
Africa                             58
Asia                               51
Europe                             50
Latin America and the Caribbean    50
Oceania                            23
Northern America                    5
Name: count, dtype: int64


**Satır satır:**
- `drop_duplicates("LocID")`: Her yer 152 kez tekrar ediyor. Bu komut her yerden sadece bir satır bırakır.
- `.set_index("LocID")`: Tabloyu LocID'ye göre aranabilir hale getirir. Bir telefon rehberi gibi düşün: numarayı ver, adı al.
- `.map(...)`: Soldaki her değeri rehberde arar ve karşılığını getirir.
  - `ulke["ParentID"].map(yerler["Location"])`: Alt bölgenin numarasını (922) adına ("Western Asia") çevirir.
  - İkinci `map` aynı işi bir basamak yukarıda yapar: Alt bölgenin üst yerini (bölgeyi) bulur.
- `drop(columns=...)`: Listedeki sütunları siler.

**🔎 Dedektif notu: Kuzey Amerika'nın 5 ülkesi.** Kod ilk denendiğinde Bermuda, Canada, Greenland, Saint Pierre and Miquelon ve ABD'nin bölgesi **boş** çıktı.
- **Sebep:** Bu ülkelerin `ParentID`'si **918**, ama dosyada 918 numaralı bir yer **yok**.
- UN'in sınıflamasında "Northern America" hem bölge hem alt bölge. Dosyada sadece bölge satırı (LocID **905**) var. Alt bölge satırı muhtemelen tekrar olmasın diye konmamış.
- **Çözüm:** Bu 5 ülkenin alt bölgesini ve bölgesini elle "Northern America" yaptık.
- **Ders:** `map` eşini bulamadığında hata vermez, **sessizce NaN yazar.** Bu yüzden her adımın sonunda `assert ... notna()` kontrolü koyuyoruz. Bu kontrol olmasaydı ABD analizden sessizce düşebilirdi.

🟡 **Soru 1:** `value_counts()` çıktısında bölgelere göre ülke sayılarına bak. En çok ülke hangi bölgede? Bu bölgede neden bu kadar çok "ülke" olabilir? **İpucu:** UN, Karayipler ve Pasifik'teki küçük adaları da "Country/Area" sayıyor.

## Adım 5 — `donem` sütunu: tahmin mi, projeksiyon mu?

**Neden?** 2023'e kadarki değerler **tahmin (estimate)**: Nüfus sayımı, kayıt ve anketlerden üretilmiş. 2024 ve sonrası **projeksiyon (projection)**: UN'in modelinin ürettiği bir senaryo. Bu ikisini her zaman ayırt edebilmeliyiz.
- Bu ayrım hem grafiklerde (dikey çizgi), hem etik bölümünde ("projeksiyon kesin bilgi değildir") kullanılacak.

In [5]:
ulke["donem"] = np.where(ulke["Time"] <= 2023, "tahmin", "projeksiyon")

print(ulke.groupby("donem")["Time"].agg(["min", "max"]))

              min   max
donem                  
projeksiyon  2024  2100
tahmin       1950  2023


**Satır satır:**
- `np.where(koşul, A, B)`: Koşul doğruysa A, yanlışsa B yazar. Excel'deki `EĞER` fonksiyonu gibi.
- `groupby("donem")["Time"].agg(["min","max"])`: Her dönemin ilk ve son yılını gösterir. Beklenen: tahmin 1950–2023, projeksiyon 2024–2100.

## Adım 6 — Ek dosyayla birleştir (merge)

**Neden?** `LE60` (60 yaşındaki birinin ortalama kaç yıl daha yaşamasının beklendiği) Medium dosyasında yok, ek dosyada var. Yaşlanma analizinde önemli bir gösterge.

**Nasıl?** İki tabloyu `LocID + Time` anahtarıyla yan yana koyacağız. Anahtar olarak `Location` kullanmıyoruz. Sebebi: Bazı isimler birden fazla LocID'de geçiyor.

**`how="left"` ne demek?** Soldaki tablonun (`ulke`) **bütün satırları korunur**. Sağdaki tabloda (`sup`) eşi olmayan satırın yeni sütunları boş (NaN) kalır. Ek dosyada Holy See (Vatikan) yok. Bu yüzden onun `LE60` değeri boş kalacak ve biz bunu bilerek kabul ediyoruz.

**Neden sadece LE60 sütunlarını alıyoruz?** Ek dosyadaki `BirthsMale` ve `BirthsFemale` yuvarlanmış sayılar. Küçük ülkelerde cinsiyet oranını bozuyorlar. Cinsiyet oranı için Medium'daki `SRB` sütununu kullanacağız (Faz 1 kararı).

In [6]:
once = len(ulke)
ulke = ulke.merge(sup[["LocID", "Time", "LE60", "LE60Male", "LE60Female"]],
                  on=["LocID", "Time"], how="left", validate="one_to_one")

print("Birleştirmeden önce:", once, "| sonra:", len(ulke))
eksik = ulke.loc[ulke["LE60"].isna(), "Location"].unique()
print("LE60'ı boş kalan ülke(ler):", eksik)

assert len(ulke) == once                    # merge satır çoğaltmadı
assert list(eksik) == ["Holy See"]          # sadece Vatikan eksik

Birleştirmeden önce: 35787 | sonra: 35787
LE60'ı boş kalan ülke(ler): ['Holy See']


**Satır satır:**
- `sup[[...]]`: Ek dosyadan sadece anahtarları ve LE60 sütunlarını alır.
- `on=["LocID","Time"]`: İki tabloyu bu iki sütun aynı olan satırlardan eşleştirir.
- `validate="one_to_one"`: "Her satırın karşı tabloda **en fazla bir** eşi olmalı" kontrolü. Anahtar yanlış seçilirse merge satırları sessizce çoğaltır. Bu, en sinsi temizlik hatalarından biridir. `validate` bu durumda hata verip durur.
- `assert len(ulke) == once`: Satır sayısının değişmediğini ayrıca kontrol eder.

🟡 **Soru 2:** Holy See'nin `LE60` değeri boş. Bu eksiklik **analizi etkiler mi**? Etkiliyorsa ne yapabiliriz? Bu sorunun cevabı Adım 8'de daha netleşecek, şimdilik tahminini yaz.

## Adım 7 — Dünya toplamıyla kontrol

**Neden?** Temizlik doğru yapıldıysa, 237 ülkenin nüfuslarının toplamı her yıl UN'in "World" satırına eşit olmalı. Bu bir **çapraz doğrulama (cross-validation of totals)**. Bir ülke eksik kalsaydı ya da bir bölge yanlışlıkla içeride olsaydı burada yakalardık.

In [7]:
toplam = ulke.groupby("Time")["TPopulation1July"].sum()
dunya  = m[m["Location"] == "World"].set_index("Time")["TPopulation1July"].loc[1950:2100]

fark_yuzde = ((toplam - dunya) / dunya * 100).abs()
print("En büyük fark (%):", round(fark_yuzde.max(), 6))
print("2023: ülkeler toplamı =", round(toplam[2023] / 1e6, 3), "milyar | World =", round(dunya[2023] / 1e6, 3), "milyar")

assert fark_yuzde.max() < 0.01

En büyük fark (%): 0.0
2023: ülkeler toplamı = 8.092 milyar | World = 8.092 milyar


**Satır satır:**
- `groupby("Time")["TPopulation1July"].sum()`: Her yıl için 237 ülkenin nüfusunu toplar.
- `dunya`: World satırının aynı yıllardaki değerleri.
- `fark_yuzde`: İkisi arasındaki farkın yüzdesi. `.abs()` mutlak değer alır.
- Nüfus **bin kişi** cinsinden. `/ 1e6` ile milyara çeviriyoruz (1e6 bin = 1 milyar).

**Sunumda:** *"After filtering to 237 countries, our totals match UN's world population in every year from 1950 to 2100."*

## Adım 8 — 🟡 K1 kararı: Hangi ülkeleri analize alacağız?

**Soru:** 237 "ülke"nin bir kısmı çok küçük. Örneğin Tokelau'nun nüfusu yaklaşık 2 bin kişi. Bu ülkeleri analizde tutalım mı?

**Neden önemli? İki tarafın da haklı gerekçeleri var:**
- **Çıkarmanın gerekçesi:**
  - Küçük nüfuslarda tek bir yıldaki birkaç doğum veya ölüm, oranları (TFR, IMR) çok oynatır. Bu da gürültü yaratır.
  - UN bu küçük yerlerin çoğu için tam hesaplama yapmaz. Değerleri genellikle modelden ve komşu ülkelerden türetir.
  - Kümelemede (Faz 5) bu tür uç değerler sonucu çarpıtabilir.
- **Tutmanın gerekçesi (etik):**
  - Küçük ada devletlerini çıkarmak, onları analizde **görünmez yapar**. Oysa iklim ve göç açısından en kırılgan ülkeler arasında yer alıyorlar.
  - ASA etik ilkeleri, hangi verinin dışlandığını ve bunun sonuca etkisini açıkça bildirmemizi istiyor.

**Üç seçenek:**
- **(a)** 237 ülkenin hepsi
- **(b)** 2023 nüfusu ≥ 100 bin
- **(c)** 2023 nüfusu ≥ 1 milyon

Önce bakalım: Her seçenekte kaç ülke kalıyor ve dünya nüfusunun yüzde kaçını kapsıyor?

In [8]:
pop2023 = ulke.loc[ulke["Time"] == 2023].set_index("LocID")["TPopulation1July"]   # bin kişi
ulke["pop2023"] = ulke["LocID"].map(pop2023)

dunya2023 = pop2023.sum()
for ad, esik in [("(a) hepsi", 0), ("(b) >= 100 bin", 100), ("(c) >= 1 milyon", 1000)]:
    kalan = pop2023[pop2023 >= esik]
    print(f"{ad:18s} → {len(kalan):3d} ülke | dünya nüfusunun %{kalan.sum() / dunya2023 * 100:.2f}'i")

(a) hepsi          → 237 ülke | dünya nüfusunun %100.00'i
(b) >= 100 bin     → 200 ülke | dünya nüfusunun %99.98'i
(c) >= 1 milyon    → 161 ülke | dünya nüfusunun %99.79'i


**Satır satır:**
- `pop2023`: Her ülkenin 2023 nüfusu (bin kişi). LocID'ye göre aranabilir.
- `ulke["pop2023"] = ...map(pop2023)`: Bu değeri her ülkenin **bütün yıllarına** bir sütun olarak yazar. Böylece seçimi ileride tek satırla uygulayabiliriz.
- `for ad, esik in [...]`: Üç seçeneği sırayla dener. Eşiğin üstünde kalan ülkeleri sayar ve dünya nüfusundaki paylarını hesaplar.
- `f"..."`: Biçimlendirilmiş yazı (f-string). `{...}` içindeki değerler metnin içine yerleştirilir. `:.2f` virgülden sonra 2 basamak demek.

In [9]:
# Hangi ülkeler çıkıyor? Bölgelere göre dağılım
kucuk = ulke.drop_duplicates("LocID").query("pop2023 < 1000")
print("1 milyonun altındaki ülkelerin bölgelere göre dağılımı:")
print(kucuk.assign(grup=np.where(kucuk["pop2023"] < 100, "< 100 bin", "100 bin – 1 milyon"))
           .groupby(["bolge", "grup"]).size().unstack(fill_value=0))
print()
print("En küçük 10 ülke:")
print(kucuk.nsmallest(10, "pop2023")[["Location", "altbolge", "pop2023"]].to_string(index=False))

1 milyonun altındaki ülkelerin bölgelere göre dağılımı:
grup                             100 bin – 1 milyon  < 100 bin
bolge                                                         
Africa                                            7          1
Asia                                              4          0
Europe                                            5          9
Latin America and the Caribbean                  13         14
Northern America                                  0          3
Oceania                                          10         10

En küçük 10 ülke:
                   Location         altbolge  pop2023
                   Holy See  Southern Europe    0.496
                       Niue        Polynesia    1.817
                    Tokelau        Polynesia    2.398
Falkland Islands (Malvinas)    South America    3.477
                 Montserrat        Caribbean    4.420
               Saint Helena   Western Africa    5.288
  Saint Pierre and Miquelon Northern Americ

**Satır satır:**
- `drop_duplicates("LocID")`: Her ülkeden tek satır bırakır.
- `.query("pop2023 < 1000")`: 1 milyonun altındaki ülkeleri seçer. Hatırla, birim bin kişi.
- `assign(grup=...)`: Geçici bir "grup" sütunu ekler: 100 binin altı mı, 100 bin ile 1 milyon arası mı?
- `groupby([...]).size().unstack(...)`: Bölge × grup için bir sayım tablosu (çapraz tablo) üretir.
- `nsmallest(10, "pop2023")`: En küçük 10 ülkeyi getirir.

🟡 **Soru 3 (K1 kararı):** Yukarıdaki iki çıktıya bak ve şunları yaz:
1. (b) ve (c) seçeneklerinde dünya nüfusunun ne kadarını kaybediyoruz? Çok mu, az mı?
2. Çıkan ülkeler **hangi bölgelerde** yoğunlaşıyor? Bu etik açıdan ne anlama geliyor?
3. Seçimin **(a), (b) veya (c)**, ve tek cümlelik gerekçen.

**Bilmen gereken önemli bir nokta:** Seçimin geri dönüşü olmayan bir karar değil. Temiz dosyada **237 ülkenin hepsini** ve `pop2023` sütununu tutuyoruz. Kapsamı Faz 5'te tek bir satırla uygulayacağız. Sonra **sağlamlık kontrolünde (robustness check)** diğer seçenekle de deneyip "sonuç değişiyor mu?" diye bakacağız. Bu yaklaşım jüriye hem istatistiksel hem etik bir güç olarak sunulabilir.

## Adım 9 — Son kontroller ve kaydetme

In [10]:
# Göstergelerde (LE60 hariç) hiç boş hücre kalmamalı
gostergeler = ulke.columns.difference(["LocID", "ISO3_code", "Location", "ParentID", "Time",
                                       "altbolge", "bolge", "donem", "pop2023",
                                       "LE60", "LE60Male", "LE60Female"])
print("Gösterge sayısı:", len(gostergeler))
print("Boş hücre:", int(ulke[gostergeler].isna().sum().sum()))
assert ulke[gostergeler].notna().all().all()

# Sütun sırası: önce kimlik, sonra göstergeler
on = ["LocID", "ISO3_code", "Location", "altbolge", "bolge", "ParentID", "Time", "donem", "pop2023"]
ulke = ulke[on + [c for c in ulke.columns if c not in on]].sort_values(["LocID", "Time"])

ulke.to_csv("../clean/countries_1950_2100.csv", index=False)
print("Kaydedildi:", ulke.shape)

Gösterge sayısı: 53
Boş hücre: 0
Kaydedildi: (35787, 65)


**Satır satır:**
- `columns.difference([...])`: Kimlik sütunları ve LE60 dışında kalan bütün sütunları, yani göstergeleri verir. Beklenen **53**: Medium'daki 54 gösterge eksi `DoublingTime`.
- `isna().sum().sum()`: Önce sütun başına, sonra tüm tablo için boş hücreleri sayar. Beklenen **0**.
- `ulke[on + [...]]`: Sütunları yeniden sıralar. Önce kimlik sütunları, sonra göstergeler.
- `sort_values(["LocID","Time"])`: Satırları önce ülkeye, sonra yıla göre sıralar.
- `to_csv(..., index=False)`: Dosyaya kaydeder. `index=False`, pandas'ın satır numaralarını ayrı bir sütun olarak yazmasını engeller.

Beklenen boyut: **35.787 satır** (237 × 151).

## Faz 2 özeti — Methods bölümüne girecek
| Adım | Ne yaptık | Kontrol |
|---|---|---|
| Ülke filtresi | `LocTypeName == "Country/Area"` → 237 ülke | ISO3 hepsinde dolu |
| Zaman | 1950–2100 (2101 çıkarıldı) | 237 × 151 satır |
| Sütunlar | `DoublingTime` ve 8 tanımlama sütunu çıkarıldı; `altbolge`, `bolge`, `donem`, `pop2023` eklendi | — |
| Merge | Ek dosyadan LE60 (`LocID + Time`, left, one-to-one) | Sadece Holy See boş |
| Dünya kontrolü | 237 ülkenin toplamı = World | Fark < %0,01 |

**Etik notları (ASA):**
- Toplam satırlar çıkarıldı (çifte sayım önlendi).
- Hiçbir ülke sessizce dışlanmadı. Kapsam kararı (K1) açıkça raporlanacak ve sağlamlık kontrolüyle sınanacak.
- Tahmin ile projeksiyon her satırda etiketli.
- Bütün adımlar `assert` ile kontrol ediliyor. Notebook baştan çalıştırıldığında aynı dosyayı üretiyor (tekrarlanabilirlik).

**Olası jüri soruları:**
- *Ülkeleri nasıl seçtiniz? Küçük ülkeleri neden dahil ettiniz ya da neden çıkardınız?*
- *İki dosyayı nasıl birleştirdiniz, satır çoğalmadığından nasıl emin oldunuz?*
- *Temizliğin doğru olduğunu nasıl doğruladınız?*

## K1 kararı: (b) 2023 nüfusu ≥ 100 bin → 200 ülke
- Dünya nüfusunun %99,98'i kapsanıyor.
- Çıkarılan 37 mikro yerin verisi büyük ölçüde UN varsayımı (ör. Holy See: TFR = 1,000).
- Çıkanlar Okyanusya (10/23) ve Karayipler'de yoğunlaşıyor; (c) seçilseydi Okyanusya'nın %87'si kaybolurdu.
- Temiz dosyada 237 ülkenin hepsi var; seçim Faz 5'te `df[df.pop2023 >= 100]` ile uygulanacak ve (c) ile sağlamlık kontrolü yapılacak.
